In [1]:
!pip install openai
!pip install langchain
!pip install langchain_community
!pip install tiktoken
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.3.11:
      Successfully uninstalled langchain-text-splitters-0.3.11
ERROR: pip's dependency resolver

In [3]:
# Import necessary libraries
import openai
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [30]:
import requests
import json

response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": "Bearer api",
        "Content-Type": "application/json"
    },
    data=json.dumps({
        "model": "deepseek/deepseek-r1",
        "messages": [
            {"role": "user", "content": "What is the a text?"}
        ]
    })
)



In [31]:
data1 = response.json()
print("Model reply:", data1["choices"][0]["message"]["content"])

Model reply: It seems there might be a small typo in your question. If you meant to ask, **"What is a text?"**, here's an explanation:

A **text** is any piece of written, spoken, or digital communication that conveys meaning. It can take many forms, such as:
- **Written content**: Books, articles, essays, messages, or even a single sentence.
- **Spoken words**: Transcripts of speeches, dialogues, or verbal communication.
- **Digital/media content**: Social media posts, emails, websites, or multimedia (e.g., subtitles, memes).
- **Symbolic systems**: Signs, codes, or cultural artifacts (e.g., a film, painting, or song lyrics).

In academic contexts, "text" often refers to material analyzed for meaning, structure, or cultural significance. For example, scholars might study a novel (*literary text*), a legal document, or even a tweet as a "text" to uncover insights.

If you meant **"What is the text?"** in reference to something specific (e.g., a passage, a quote, or a work), please prov

In [12]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.6/323.6 kB 6.2 MB/s eta 0:00:00


In [17]:
!pip install docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for docx: filename=docx-0.2.4-py3-none-any.whl size=53893 sha256=afa635f5183395406e7217b77bc67a267fb845f2928b76b359ebd7542fa2e855
  Stored in directory: /root/.cache/pip/wheels/f3/ba/dd/43ed5f165600f41deddeb1e382c56ffc1067c09ec5bd705f39
Successfully built docx


###  Using Custom data

In [2]:
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import PyPDF2, docx
from google.colab import files


In [3]:
api_key = ""
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")


In [4]:
print("Upload your document (PDF, DOCX, or TXT):")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
print("Uploaded:", fname)


ext = fname.lower().split('.')[-1]
if ext == "pdf":
    text = "\n".join(page.extract_text() for page in PyPDF2.PdfReader(fname).pages)
elif ext == "docx":
    text = "\n".join(p.text for p in docx.Document(fname).paragraphs)
elif ext == "txt":
    text = open(fname, "r", encoding="utf-8").read()
else:
    raise ValueError("Unsupported file type: " + ext)


Upload your document (PDF, DOCX, or TXT):


Saving RAGPaper+(1).pdf to RAGPaper+(1) (2).pdf
Uploaded: RAGPaper+(1) (2).pdf


In [5]:
import re


sentences = re.split(r'(?<=[.!?\n])', text)  # split at . ! ? or newline
sentences = [s.strip() for s in sentences if s.strip()]


chunk_size = 500
chunks = []
current_chunk = ""

for sent in sentences:
    if len(current_chunk) + len(sent) + 1 <= chunk_size:
        current_chunk += " " + sent
    else:
        chunks.append(current_chunk.strip())
        current_chunk = sent

# Add the last chunk
if current_chunk:
    chunks.append(current_chunk.strip())

print("Total chunks:", len(chunks))


Total chunks: 148


In [6]:
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2), stop_words='english')
embeddings = vectorizer.fit_transform(chunks)
print("TF-IDF embeddings created.")


TF-IDF embeddings created.


In [10]:
question = "who is the author of this "

q_vec = vectorizer.transform([question])
sims = cosine_similarity(q_vec, embeddings).flatten()
top_k = 3
top_idx = np.argsort(sims)[-top_k:][::-1]
context = "\n\n".join(chunks[i] for i in top_idx)
print("Context for the question:\n", context[:500], "…")


Context for the question:
 ﬁnd evidence for this hypothesis by feeding the BART-only baseline with the partial decoding "The Sun. BART completes the generation "The SunAlso Rises" isanovel bythis author of"The Sun Also Rises" indicating the title "The Sun Also Rises" is stored in BART’s parameters. Similarly, BART will complete the partial decoding "The SunAlso Rises" isanovel bythis author of"A with "The SunAlso Rises" isanovel bythis author of"AFarewell toArms" . This example shows

I Number of instances per dataset The …


In [11]:
model = "deepseek/deepseek-r1"  # check with the provider if the prefix is needed
prompt = f"""Based on the following document excerpts, answer the question:

Context:
{context}

Question: {question}

If the answer cannot be found in the document, say so."""

response = client.chat.completions.create(
    model=model,
    messages=[{"role":"user","content":prompt}],
    temperature=0.6
)
answer = response.choices[0].message.content
print("Answer:", answer)


Answer: The answer cannot be found in the document. While the excerpts indicate that BART associates the titles "The Sun Also Rises" and "A Farewell to Arms" with their respective authors (via partial decoding examples), the document does not explicitly name the author (e.g., Ernest Hemingway). The provided context focuses on the model's behavior and technical details rather than identifying the author.
